In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import cv2
import os

from plotting_utils import (
    plot_results,
    plot_first_order_grads,
    plot_second_order_grads,
    plot_change_in_value_per_stripe,
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


def compute_radial_loss(phase_map, center_h, center_w, target_wavelength=10.0, w_monotonicity=1.0, w_frequency=1.0, w_low_pixel_value=0.01):
    """
    phase_map: (Batch, H, W) or (H, W) tensor of continuous phase values.
    """
    if phase_map.dim() == 2:
        phase_map = phase_map.unsqueeze(0)

    h, w = phase_map.shape[-2:]

    y = torch.arange(h).to(phase_map.device).float()
    x = torch.arange(w).to(phase_map.device).float()
    grid_y, grid_x = torch.meshgrid(y, x, indexing="ij")

    vec_y = grid_y - center_h
    vec_x = grid_x - center_w

    dist = torch.sqrt(vec_y**2 + vec_x**2 + 1e-8)
    unit_y = vec_y / dist
    unit_x = vec_x / dist

    dx = phase_map[:, :, 1:] - phase_map[:, :, :-1]
    dy = phase_map[:, 1:, :] - phase_map[:, :-1, :]

    dy = F.pad(dy, (0, 0, 0, 1))  # Pad bottom row
    dx = F.pad(dx, (0, 1, 0, 0))  # Pad right col

    radial_gradient = (dy * unit_y) + (dx * unit_x)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(phase_map[0].detach().cpu().numpy(), cmap="gray")
    plt.colorbar()
    plt.subplot(1, 2, 2)
    plt.imshow(radial_gradient[0].detach().cpu().numpy(), cmap="gray")
    plt.colorbar()
    # plt.imshow(vec_x + vec_y, cmap="RdBu")
    plt.show()

    loss_monotonic = torch.mean(-radial_gradient)
    
    grad_mag = torch.sqrt(dx**2 + dy**2 + 1e-8)
    target_slope = (2 * math.pi) / target_wavelength
    loss_frequency = torch.mean((grad_mag - target_slope) ** 2)

    loss_sparsity = torch.mean(torch.abs(phase_map))

    return (
        (w_monotonicity * loss_monotonic)
        + (w_frequency * loss_frequency)
        + (w_low_pixel_value * loss_sparsity)
    )

def train_continuity_loss(target_image, n_epochs, spiral_phase_map=None, target_wavelength=10.0):
    h, w = target_image.shape
    phase_map = torch.zeros_like(target_image, requires_grad=True)

    if spiral_phase_map is None:
        spiral_phase_map = torch.zeros_like(target_image, requires_grad=False)

    # training settings
    optimizer = torch.optim.Adam([phase_map], lr=0.1)

    center_y, center_x = h // 2, w // 2

    for epoch in range(n_epochs):
        optimizer.zero_grad()
        generated_fingerprint = 0.5 * (1.0 - torch.cos(phase_map + spiral_phase_map))
        # normalize generated fingerprint
        generated_fingerprint = generated_fingerprint / max(generated_fingerprint.max(), -generated_fingerprint.min())

        # Compute continuity loss
        continuity_loss = compute_radial_loss(
            phase_map,
            center_y,
            center_x,
            target_wavelength=target_wavelength,
            w_monotonicity=3.0,
            w_frequency=0.0,
            w_low_pixel_value=0.0,
        )

        recon_loss = torch.nn.MSELoss()(generated_fingerprint, target_image)
        zero_at_center = torch.pow(torch.abs(phase_map[center_y, center_x]), 2)

        # Total loss
        total_loss = continuity_loss + 0.5 * recon_loss + 10.0 * zero_at_center

        # Backpropagate and update parameters
        total_loss.backward()
        optimizer.step()

        # Print loss values
        if epoch % 100 == 0 or epoch == 1 or epoch == n_epochs - 1:
            print(
                f" Step {epoch}: Rec Loss: {recon_loss.item():.6f}, Continuity Loss: {continuity_loss:.6f}, Total Loss: {total_loss.item():.6f}"
            )
    
    return phase_map

In [ ]:
import torch
import numpy as np


def train_ring_by_ring(
    target_image,
    total_rings = 10,
    n_epochs = 500,
    spiral_phase_map = None
):
    # Initialize the continuous phase (e.g., all zeros or a flat plane)
    # We will learn the "height map" iteratively.
    h, w = target_image.shape

    phase_map = torch.zeros_like(target_image, requires_grad=True)

    if spiral_phase_map is None:
        spiral_phase_map = torch.zeros_like(target_image, requires_grad=False)

    # training settings
    optimizer = torch.optim.Adam([phase_map], lr=0.1)

    center_y, center_x = h // 2, w // 2

    # Pre-compute distance map for rings
    y, x = torch.meshgrid(torch.arange(h), torch.arange(w))
    dist = torch.sqrt((x - center_x) ** 2 + (y - center_y) ** 2)
    max_dist = dist.max()

    for k in range(total_rings):
        # define Radial Limits for this step
        r_inner = (k * max_dist) / total_rings
        r_outer = ((k + 1) * max_dist) / total_rings

        mask_active = (dist >= r_inner) & (dist < r_outer)

        # Border: The seam where the new ring touches the old ring
        # We need this to force continuity with the past.
        # (For k=0, there is no border).
        if k > 0:
            mask_border = (dist >= r_inner - 2) & (dist < r_inner + 2)
        else:
            mask_border = torch.zeros_like(mask_active)

        print(f"Optimizing Ring {k}: Radius {r_inner:.1f} to {r_outer:.1f}")
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.axis("off")
        plt.imshow(mask_active.numpy(), cmap="gray")

        plt.subplot(1, 3, 2)
        plt.imshow(mask_border.numpy(), cmap="Reds", alpha=0.5)
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.axis("off")
        plt.imshow(phase_map.detach().numpy(), cmap="gray", alpha=0.3)
        plt.colorbar()

        plt.show()

        for epoch in range(n_epochs):
            optimizer.zero_grad()

            generated_fingerprint = 0.5 * (
                1.0 - torch.cos(phase_map + spiral_phase_map)
            )
            
            rec_loss = torch.mean(
                (generated_fingerprint[mask_active] - target_image[mask_active]) ** 2
            )

            # We enforce that gradients at the border are smooth (no cliffs).
            # If the inner ring ends at 10.0 and outer starts at 0.0,
            # the gradient here will be huge (10.0). We punish that.
            if k > 0:
                # Calculate local gradients at the border
                # (Simple finite difference or Sobel)
                grad_y = phase_map[1:, :] - phase_map[:-1, :]
                grad_x = phase_map[:, 1:] - phase_map[:, :-1]

                grad_x = F.pad(grad_x, (0, 1, 0, 0))
                grad_y = F.pad(grad_y, (0, 0, 0, 1))

                grad_magnitude = torch.sqrt(grad_x**2 + grad_y**2 + 1e-8)
                border_grads = grad_magnitude[mask_border]

                # Penalize "Jumps" at the border
                continuity_loss = torch.mean(torch.abs(border_grads))
            else:
                continuity_loss = 0.0

            total_loss = rec_loss + 0.5 * continuity_loss
            total_loss.backward()

            # only update grads in active ring
            phase_map.grad = phase_map.grad * mask_active.float()

            optimizer.step()

            if epoch % 100 == 0 or epoch == 1 or epoch == n_epochs - 1:
                print(
                    f" Step {epoch}: Rec Loss: {rec_loss.item():.6f}, Continuity Loss: {continuity_loss if k > 0 else 0.0:.6f}, Total Loss: {total_loss.item():.6f}"
                )

    return phase_map

In [ ]:
img = cv2.imread("../images/spiral_phase.jpg", cv2.IMREAD_GRAYSCALE)
img = img / 255.0
h, w = img.shape
target_image = torch.from_numpy(img).float()
# normalize target image
target_image = target_image / target_image.max()

plt.imshow(target_image.numpy(), cmap="gray")
plt.axis("off")
plt.colorbar()
plt.show()
# spiral phase initialization
spiral_phase_coords=[(95, 128), (40, 55), (128, 200)]
spiral_phase_polarities=[+1, -1, -1]

spiral_phase = torch.zeros(h, w)
y_range = torch.arange(h)
x_range = torch.arange(w)

Y, X = torch.meshgrid(y_range, x_range, indexing="ij")
for (yy, xx), polarity in zip(spiral_phase_coords, spiral_phase_polarities):
    spiral_phase += polarity * torch.arctan2(Y - yy, X - xx)

spiral_phase_map = torch.tensor(spiral_phase, requires_grad=False)

# trained_phase_map = train_ring_by_ring(
#     target_image,
#     total_rings=10,
#     n_epochs=500,
#     spiral_phase_map=spiral_phase_map
# )
trained_phase_map = train_continuity_loss(target_image, 100, spiral_phase_map=spiral_phase_map, target_wavelength=2 * math.pi / 0.32)

final_trained_fingerprint = 0.5 * (
    1.0 - torch.cos(trained_phase_map + spiral_phase_map)
)

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Target Image")
plt.imshow(target_image.detach().cpu(), cmap="gray")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Generated Fingerprint")
plt.imshow(final_trained_fingerprint.detach().cpu(), cmap="gray")
plt.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(trained_phase_map.detach().cpu(), cmap="gray")
plt.colorbar()
plt.title("Trained Phase Map")
plt.axis("off")

plt.subplot(1, 2, 2)
final_fingerprint_without_spiral = 0.5 * (
    1.0 - torch.cos(trained_phase_map)
)
plt.imshow(final_fingerprint_without_spiral.detach().cpu(), cmap="gray")
plt.title("Generated Fingerprint without Spiral Phase")
plt.axis("off")

plt.show()
plt.imshow(spiral_phase_map.detach().cpu(), cmap="gray")
plt.colorbar()
plt.title("Spiral Phase Map")
plt.axis("off")
plt.show()